Partition and move criteo raw data to s3

In [ ]:
from datasets import load_dataset
import os

train_file_path = "../../../datasets/criteo_attribution_dataset"

full = load_dataset(
    'csv',
    data_files=os.path.join(train_file_path, "criteo_attribution_dataset.tsv"),
    delimiter='\t'
)
full

In [5]:
from sklearn.preprocessing import OrdinalEncoder
import joblib

# list features that will require embeddings and print their cardinality
sparse_features = ["uid", "campaign"]+[f"cat{i}" for i in range(1, 10)]
sparse_voc_size = list()
V = dict()
for f in sparse_features:
    unique_values = full["train"].unique(f)
    
    
    print(f, len(unique_values))
    sparse_voc_size.append(len(unique_values))
    V[f] = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=len(unique_values)+1, dtype=int)
    V[f].fit([[val] for val in unique_values])

joblib.dump(V, "assets/ray/ordinal_encoders.joblib", compress=3)
print(sparse_features)


uid 6142256
campaign 675
cat1 9
cat2 70
cat3 1829
cat4 21
cat5 51
cat6 30
cat7 57196
cat8 11
cat9 30
['uid', 'campaign', 'cat1', 'cat2', 'cat3', 'cat4', 'cat5', 'cat6', 'cat7', 'cat8', 'cat9']


In [13]:
V = joblib.load("assets/ray/ordinal_encoders.joblib")

uid_encoder = V["uid"]
encoded = uid_encoder.transform([[19258185]])
print(encoded)

[[3644340]]


In [ ]:
import s3fs
import polars as pl
from datetime import datetime, timezone

# import math

S3_BUCKET = "<MY-BUCKET>"
S3_PREFIX = "criteo/unprocessed"
# N_SHARDS = 20

fs = s3fs.S3FileSystem()

dataset = full["train"]
# shard_size = math.ceil(len(dataset) / N_SHARDS)

# for i in range(N_SHARDS):
#     start = i * shard_size
#     end = min(start + shard_size, len(dataset))
#     shard = dataset.select(range(start, end))
#     s3_path = f"s3://{S3_BUCKET}/{S3_PREFIX}/part-{i:05d}.parquet"
#     with fs.open(s3_path, "wb") as f:
#         shard.to_parquet(f)
#     print(f"[{i+1}/{N_SHARDS}] written {len(shard)} rows → {s3_path}")

now = datetime.now(timezone.utc).replace(tzinfo=None)
# fast vectorized grouping
df = dataset.to_polars()

# create timestamp to represent an actual timestamp, criteo default timestamp is a relative number
df = df.with_columns(
    (pl.lit(now) + pl.duration(seconds=pl.col("timestamp") - df["timestamp"].max()))
    # .dt.strftime("%Y-%m-%d %H:%M:%S")
    .dt.replace_time_zone(None)
    .cast(pl.Datetime("us"))
    .alias("event_timestamp")
)

# Add year, month, day columns derived from the timestamp
df = df.with_columns([
    pl.col("event_timestamp").dt.year().alias("year"),
    pl.col("event_timestamp").dt.month().alias("month"),
    pl.col("event_timestamp").dt.day().alias("day"),
])

# Partition by year, month, day
partitions = df.partition_by(["year", "month", "day"], as_dict=True)

for keys, part in partitions.items():
    year, month, day = keys
    s3_path = f"s3://{S3_BUCKET}/{S3_PREFIX}/year={year}/month={month:02d}/day={day:02d}/data.parquet"
    with fs.open(s3_path, "wb") as f:
        part.drop(["year", "month", "day"]).write_parquet(
            f,
            use_pyarrow=True,
            pyarrow_options={
                "coerce_timestamps": "us",
                "allow_truncated_timestamps": True,
            }
        )
    print(f"year={year} month={month} day={day}: {len(part)} rows → {s3_path}")


In [32]:
df.schema

Schema([('timestamp', Int64),
        ('uid', Int64),
        ('campaign', Int64),
        ('conversion', Int64),
        ('conversion_timestamp', Int64),
        ('conversion_id', Int64),
        ('attribution', Int64),
        ('click', Int64),
        ('click_pos', Int64),
        ('click_nb', Int64),
        ('cost', Float64),
        ('cpo', Float64),
        ('time_since_last_click', Int64),
        ('cat1', Int64),
        ('cat2', Int64),
        ('cat3', Int64),
        ('cat4', Int64),
        ('cat5', Int64),
        ('cat6', Int64),
        ('cat7', Int64),
        ('cat8', Int64),
        ('cat9', Int64),
        ('event_timestamp', Datetime(time_unit='us', time_zone=None)),
        ('year', Int32),
        ('month', Int8),
        ('day', Int8)])

In [31]:
df.head()

timestamp,uid,campaign,conversion,conversion_timestamp,conversion_id,attribution,click,click_pos,click_nb,cost,cpo,time_since_last_click,cat1,cat2,cat3,cat4,cat5,cat6,cat7,cat8,cat9,event_timestamp,year,month,day
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,datetime[μs],i32,i8,i8
0,20073966,22589171,0,-1,-1,0,0,-1,-1,0.00001,0.390794,-1,5824233,9312274,3490278,29196072,11409686,1973606,25162884,29196072,29196072,2026-02-20 22:48:16.738575,2026,2,20
2,24607497,884761,0,-1,-1,0,0,-1,-1,0.00001,0.0596,423858,30763035,9312274,14584482,29196072,11409686,1973606,22644417,9312274,21091111,2026-02-20 22:48:18.738575,2026,2,20
2,28474333,18975823,0,-1,-1,0,0,-1,-1,0.000183,0.149706,8879,138937,9312274,10769841,29196072,5824237,138937,1795451,29196072,15351056,2026-02-20 22:48:18.738575,2026,2,20
3,7306395,29427842,1,1449193,3063962,0,1,0,7,0.000094,0.154785,-1,28928366,26597095,12435261,23549932,5824237,1973606,9180723,29841067,29196072,2026-02-20 22:48:19.738575,2026,2,20
3,25357769,13365547,0,-1,-1,0,0,-1,-1,0.000032,0.037583,-1,138937,26597094,31616034,29196072,11409684,26597096,4480345,29196072,29196072,2026-02-20 22:48:19.738575,2026,2,20


In [1]:
# from datasets import Dataset

# data_path = "<MY-BUCKET>/feast/criteo/transformed/features/year=2026/month=03/day=20/*"
# transformed_df = Dataset.from_parquet(f"s3://{data_path}")
# transformed_pl = transformed_df.to_polars()